# 08 — Full Evaluation Matrix

Runs the full **2 benchmarks × 7 scorers × 3 router modes × 250 examples** matrix with 9 metrics per cell plus an EFL first-error-type histogram. Every cell saves its own checkpoint to `data/results/cells/` and skips on resume.

Runs on a Colab T4/A100 instance in roughly 4–5 hours. Each section below is independently resumable — if the notebook crashes, re-running this cell re-enters the matrix at the first missing cell.

## 1. Setup

In [ ]:
import sys, os

# ── Colab: mount Drive + install package ────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # --- Configure paths ---
    # Edit DRIVE_ROOT to match where you uploaded the repo on your Drive.
    DRIVE_ROOT = '/content/drive/MyDrive/HaluGuard'
    os.environ['HALUGUARD_ROOT']     = DRIVE_ROOT
    os.environ['HALUGUARD_DATA_DIR'] = f'{DRIVE_ROOT}/data'

    # --- Clone or pull the repo into /content/HaluGuard ---
    REPO_URL = os.environ.get('HALUGUARD_REPO_URL', '')  # set this if cloning from git
    if os.path.isdir('/content/HaluGuard/.git'):
        print('repo already present — pulling latest')
        os.system('git -C /content/HaluGuard pull --ff-only')
    elif REPO_URL:
        os.system(f'git clone {REPO_URL} /content/HaluGuard')
    else:
        # Fallback: copy from Drive if you uploaded the folder there
        if not os.path.isdir('/content/HaluGuard'):
            os.system(f'cp -r {DRIVE_ROOT} /content/HaluGuard')

    # --- Install haluguard + metric deps ---
    os.system('pip install -q -e /content/HaluGuard')
    os.system('pip install -q sacrebleu rouge_score codebleu')

    # Add to path so imports resolve immediately without kernel restart
    if '/content/HaluGuard' not in sys.path:
        sys.path.insert(0, '/content/HaluGuard')

    print('Colab setup complete')
else:
    # ── Local / already-installed environment ──────────────────────────────
    # Make sure the repo root is on sys.path when running the notebook
    # from inside notebooks/ (jupyter starts with notebooks/ as cwd).
    repo_root = str(Path.cwd().parent)
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)

    # Install metric deps if missing
    os.system('pip install -q sacrebleu rouge_score codebleu')
    print('Local setup complete')

# Quick import check
import haluguard  # noqa: F401
print('haluguard imported OK from', haluguard.__file__)


In [ ]:
import os, random, json
from pathlib import Path

import numpy as np
import torch

random.seed(0); np.random.seed(0); torch.manual_seed(0)

ROOT = Path(os.environ.get('HALUGUARD_ROOT', Path.cwd().parent))
DATA = Path(os.environ.get('HALUGUARD_DATA_DIR', ROOT / 'data'))
RESULTS = DATA / 'results'
CELLS = RESULTS / 'cells'
EMBEDS = DATA / 'embeddings'
CKPT = ROOT / 'checkpoints'
for d in (DATA, RESULTS, CELLS, EMBEDS, CKPT):
    d.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device =', DEVICE)
print('results dir =', RESULTS)

## 2. Load encoder + generator (Qwen2.5-Coder-7B by default)

In [ ]:
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM

ENCODER_NAME = 'microsoft/codebert-base'
GEN_NAME = os.environ.get('HALUGUARD_GEN_NAME', 'Qwen/Qwen2.5-Coder-7B')

enc_tok = AutoTokenizer.from_pretrained(ENCODER_NAME)
encoder = AutoModel.from_pretrained(ENCODER_NAME).to(DEVICE).eval()
for p in encoder.parameters():
    p.requires_grad_(False)

gen_tok = AutoTokenizer.from_pretrained(GEN_NAME, trust_remote_code=True)
gen_model = AutoModelForCausalLM.from_pretrained(
    GEN_NAME, torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
    device_map='auto', trust_remote_code=True,
).eval()
if gen_tok.pad_token_id is None:
    gen_tok.pad_token_id = gen_tok.eos_token_id

## 3. Smoke gate — 10 examples per benchmark.
Halts the notebook if any metric is NaN or EM falls outside the sanity band.

In [ ]:
from haluguard.benchmarks.repobench import RepoBenchLoader
from haluguard.benchmarks.crosscodeeval import CrossCodeEvalLoader
from haluguard.pipeline import HaluGuardPipeline
from haluguard.type_router import RegexTypeRouter
from haluguard.run_eval import run_benchmark, build_generator_from_model
from haluguard.models import MODEL_REGISTRY

GEN = build_generator_from_model(gen_tok, gen_model, device=DEVICE)

def build_pipeline(scorer_name: str, router):
    scorer = MODEL_REGISTRY[scorer_name]().to(DEVICE).eval()
    ckpt = CKPT / f'{scorer_name}_best.pt'
    if ckpt.exists():
        scorer.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    return HaluGuardPipeline.from_trained(
        scorer=scorer, tokenizer=enc_tok, encoder=encoder,
        device=DEVICE, scorer_name=scorer_name, router=router,
    )

smoke_loader = RepoBenchLoader()
smoke_pipe = build_pipeline('dual_encoder', RegexTypeRouter())
smoke = run_benchmark(
    loader=smoke_loader, pipeline=smoke_pipe, generator=GEN,
    metrics=['em','es','codebleu','bleu_4','chrf_plus','rouge_l','identifier_em','identifier_f1','codebert_score'],
    limit=10, batch_size=8, use_efl=False,
    cell_id='smoke__repobench__dual_encoder__regex', resume_dir=CELLS,
)
assert all(np.isfinite(v) for v in smoke['metrics'].values()), smoke
print('SMOKE OK', smoke['metrics'])

## 4. (Optional) Collect EFL traces on RepoBench-train for the learned router
Skip if `data/router_traces.jsonl` already exists.

In [ ]:
TRACES_PATH = DATA / 'router_traces.jsonl'
if not TRACES_PATH.exists():
    from datasets import load_dataset
    from haluguard.run_eval import _run_efl_for_example
    from haluguard.benchmarks.repobench import _row_to_example

    train_ds = load_dataset('tianyang/repobench_python_v1.1', split='cross_file_first')
    def gen_single(prompt):
        return GEN([prompt])[0]
    pipe = smoke_pipe  # reuse dual_encoder + regex for trace collection
    with TRACES_PATH.open('w') as fh:
        written = 0
        for i, row in enumerate(train_ds):
            if written >= 3000:
                break
            ex = _row_to_example(row, i)
            if ex is None:
                continue
            try:
                _, _, efl = _run_efl_for_example(pipe, ex, gen_single, max_iterations=1, timeout=6)
                err = efl.history[0].error_type if efl.history else None
            except Exception:
                err = None
            fh.write(json.dumps({'cropped_code': ex.cropped_code, 'error_type': err}) + '\n')
            written += 1
    print('wrote', written, 'traces')
else:
    print('traces already exist:', TRACES_PATH)

## 5. Train the LearnedTypeRouter

In [ ]:
from haluguard.training import RouterTrace, train_learned_router
from haluguard.type_router import LearnedTypeRouter

ROUTER_CKPT = CKPT / 'learned_router.pt'
if ROUTER_CKPT.exists():
    learned_router = LearnedTypeRouter.load(ROUTER_CKPT, encoder=encoder, tokenizer=enc_tok, device=DEVICE)
else:
    traces = []
    with TRACES_PATH.open() as fh:
        for line in fh:
            rec = json.loads(line)
            traces.append(RouterTrace(cropped_code=rec['cropped_code'], error_type=rec.get('error_type')))
    learned_router = train_learned_router(traces, encoder=encoder, tokenizer=enc_tok, device=DEVICE)
    learned_router.save(ROUTER_CKPT)
print('learned router ready @', ROUTER_CKPT)

## 6. Main matrix loop

In [ ]:
from haluguard.type_router import NoOpTypeRouter

SCORERS = ['dual_encoder','dual_encoder_deep','listwise_mlp','pairwise_mlp','interaction_mlp','bilinear','ensemble']
ROUTER_BUILDERS = {
    'regex':  lambda: RegexTypeRouter(),
    'noop':   lambda: NoOpTypeRouter(),
    'learned': lambda: learned_router,
}
BENCHMARKS = {
    'repobench':     lambda: RepoBenchLoader(),
    'crosscodeeval': lambda: CrossCodeEvalLoader(),
}
METRICS = ['em','es','codebleu','bleu_4','chrf_plus','rouge_l','identifier_em','identifier_f1','codebert_score']
LIMIT = int(os.environ.get('HALUGUARD_LIMIT', 250))

summaries = []
for bname, build_loader in BENCHMARKS.items():
    loader = build_loader()
    for scorer_name in SCORERS:
        for rname, build_router in ROUTER_BUILDERS.items():
            cell_id = f'{bname}__{scorer_name}__{rname}'
            out_json = CELLS / f'{cell_id}.json'
            if out_json.exists():
                summaries.append(json.loads(out_json.read_text()))
                print('skip (done):', cell_id)
                continue
            router = build_router()
            pipe = build_pipeline(scorer_name, router)
            summary = run_benchmark(
                loader=loader, pipeline=pipe, generator=GEN,
                metrics=METRICS, limit=LIMIT, batch_size=16,
                use_efl=False, cell_id=cell_id, resume_dir=CELLS,
            )
            summaries.append(summary)
            del pipe
            torch.cuda.empty_cache() if DEVICE == 'cuda' else None
print('matrix done —', len(summaries), 'cells')

## 7. Aggregate into `results_table.json` + `results_summary.md`

In [ ]:
rows = []
for f in sorted(CELLS.glob('*.json')):
    rec = json.loads(f.read_text())
    row = {'cell_id': rec['cell_id'], 'benchmark': rec['benchmark'],
           'scorer': rec['scorer'], 'router': rec['router'], 'n': rec['n']}
    row.update(rec.get('metrics', {}))
    rows.append(row)

(table_path := RESULTS / 'results_table.json').write_text(json.dumps(rows, indent=2, default=str))
print('wrote', table_path, 'with', len(rows), 'rows')

# Quick ranked summary by EM per benchmark.
lines = ['# HaluGuard evaluation results', '']
for bname in sorted({r['benchmark'] for r in rows}):
    lines.append(f'## {bname}')
    sub = sorted([r for r in rows if r['benchmark'] == bname], key=lambda r: -r.get('em', 0.0))
    hdr = ['scorer','router','n','em','es','codebleu','bleu_4','chrf_plus','rouge_l','identifier_em','identifier_f1','codebert_score']
    lines.append('| ' + ' | '.join(hdr) + ' |')
    lines.append('|' + '|'.join(['---'] * len(hdr)) + '|')
    for r in sub:
        lines.append('| ' + ' | '.join(f'{r.get(c, ""):.4f}' if isinstance(r.get(c), float) else str(r.get(c, '')) for c in hdr) + ' |')
    lines.append('')
(RESULTS / 'results_summary.md').write_text('\n'.join(lines))
print('wrote', RESULTS / 'results_summary.md')